In [1]:
from pathlib import Path
notebook_dir = Path().resolve()
data_path = notebook_dir.parent / "DataSet" / "secom.csv"
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.api as sm
import datetime as dt
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Microsoft JhengHei'  # or 'Noto Sans TC'
plt.rcParams['axes.unicode_minus'] = False
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from scipy import stats as sps
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.impute import KNNImputer
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, accuracy_score, f1_score, r2_score, mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.stattools import durbin_watson, jarque_bera
from statsmodels.stats.diagnostic import linear_reset, linear_rainbow, het_breuschpagan
from xgboost import XGBClassifier


In [2]:
secom = pd.read_csv(data_path, sep='\t')

X = secom.drop(columns=['Time', 'Pass/Fail'])
y = secom['Pass/Fail'].replace({-1:0, 1:1}).astype(int)

na_counts = X.isna().sum()
X = X.loc[:, na_counts <= 1000].copy()
uniq_counts = X.nunique(dropna=False)
X = X.loc[:, uniq_counts >= 10].copy()
X = X.loc[:, ~X.T.duplicated(keep='first')].copy()

imputer = KNNImputer(n_neighbors=10, weights='distance')
X_imp = pd.DataFrame(imputer.fit_transform(X), columns=X.columns, index=X.index)

scaler = StandardScaler()
X_std = pd.DataFrame(scaler.fit_transform(X_imp), columns=X.columns, index=X.index)

pca = PCA(n_components=0.9)
X_pca = pca.fit_transform(X_std)
print(f'PCA 保留 {X_pca.shape[1]} 個主成分')
X_pca_df = pd.DataFrame(X_pca, columns=[f'pc{i}' for i in range(1, 132)])

PCA 保留 131 個主成分


---
---

In [4]:
def _sample_offsets_block(q, k, n_block, center_first=True, rng=None):
    """
    產生 (n_block, q) 的整數位移 offsets ∈ [-k, k]
    center_first=True：第一筆固定為全 0
    """
    if rng is None:
        rng = np.random.default_rng(0)

    offsets = rng.integers(-k, k + 1, size=(n_block, q), dtype=np.int16).astype(np.float32)
    if center_first and n_block > 0:
        offsets[0, :] = 0.0
    return offsets

def _ols_from_sufficient_stats(Sxx, Sxy, Syy, sum_y, n, q):
    """
    由 sufficient statistics（X'X, X'y, y'y, sum(y)）計算 OLS 係數與統計量
    用累積量求 OLS 係數與統計量
    輸出包含 beta / se / t / p / r2 / aic / bic / df_resi
    """
    p_params = q + 1
    df_resid = n - p_params
    out = {}

    try:
        beta = np.linalg.solve(Sxx, Sxy)
    except np.linalg.LinAlgError:
        beta = np.full(p_params, np.nan, dtype=np.float64)

    if np.all(np.isfinite(beta)):
        SSE = float(Syy - 2.0 * beta @ Sxy + beta @ (Sxx @ beta))
    else:
        SSE = np.nan

    ybar = float(sum_y / n) if n > 0 else np.nan
    TSS = float(Syy - n * (ybar**2)) if n > 0 else np.nan
    sigma2 = float(SSE / df_resid) if (df_resid > 0 and np.isfinite(SSE)) else np.nan

    try:
        Sxx_inv = np.linalg.inv(Sxx)
    except np.linalg.LinAlgError:
        Sxx_inv = np.full_like(Sxx, np.nan, dtype=np.float64)

    if np.isfinite(sigma2) and np.all(np.isfinite(Sxx_inv)):
        cov = sigma2 * Sxx_inv
        se = np.sqrt(np.diag(cov))
        tval = beta / se
        pval = 2.0 * sps.t.sf(np.abs(tval), df=df_resid) if df_resid>0 else np.full_like(tval, np.nan)
    else:
        se = np.full(p_params, np.nan, dtype=np.float64)
        tval = np.full(p_params, np.nan, dtype=np.float64)
        pval = np.full(p_params, np.nan, dtype=np.float64)

    r2 = (1.0 - SSE / TSS) if (np.isfinite(SSE) and np.isfinite(TSS) and TSS > 0) else np.nan

    kparam = p_params
    if n > 0 and np.isfinite(SSE) and SSE > 0:
        ll_term = n * np.log(SSE / n)
        aic = float(ll_term + 2.0 * kparam)
        bic = float(ll_term + np.log(n) * kparam)
    else:
        aic = np.nan
        bic = np.nan

    out["beta"] = beta
    out["SSE"] = SSE
    out["r2"] = float(r2) if np.isfinite(r2) else np.nan
    out["se"] = se
    out["t"] = tval
    out["p"] = pval
    out["aic"] = aic
    out["bic"] = bic
    out["df_resid"] = int(df_resid)
    return out


def local_surrogate_block_sample_ols(
    x0,
    blackbox_model,
    block_cols,
    delta,
    k,
    scaler=None,
    n_block=800_000,
    rng_seed=42,
    store_yhat=False,
    store_yblackbox=False,
):
    """
    在 x0 附近抽樣 n_block 個格點，丟進黑箱得到 y_hat
    一次只用 block_cols 的 q 維特徵做 OLS，輸出係數與 p-value
    """
    x0 = np.asarray(x0, np.float32).reshape(1, -1)

    if scaler is not None:
        x0_scaled = scaler.transform(x0).astype(np.float32, copy=False)
    else:
        x0_scaled = x0.copy()

    p_model = x0_scaled.shape[1]
    block_cols = np.asarray(block_cols, dtype=np.int64)
    q = int(block_cols.size)
    if q < 1:
        raise ValueError("block_cols 至少 1 維")
    if k < 0:
        raise ValueError("k must be >= 0")
    if store_yhat or store_yblackbox:
        raise ValueError("store_yhat/store_yblackbox 在大 n 下不可用（請關掉）。")

    rng = np.random.default_rng(rng_seed)

    offsets_q = _sample_offsets_block(q, k, n_block, center_first=True, rng=rng)
    X_block = (np.float32(delta) * offsets_q).astype(np.float32, copy=False)

    Zs = np.empty((n_block, p_model), dtype=np.float32)
    Zs[:] = x0_scaled
    Zs[:, block_cols] += X_block

    if scaler is not None:
        Z = scaler.inverse_transform(Zs).astype(np.float32, copy=False)
    else:
        Z = Zs

    if hasattr(blackbox_model, "predict_proba"):
        y = blackbox_model.predict_proba(Z)[:, 1]
    else:
        y = blackbox_model.predict(Z)

    y = np.asarray(y, dtype=np.float32).reshape(-1)
    if y.size != n_block:
        raise ValueError("blackbox 輸出長度與 n_block 不一致")

    Sxx = np.zeros((q + 1, q + 1), dtype=np.float64)
    Sxy = np.zeros((q + 1,), dtype=np.float64)

    Sxx[0, 0] = float(n_block)
    sx = X_block.sum(axis=0, dtype=np.float64)
    Sxx[0, 1:] = sx
    Sxx[1:, 0] = sx
    Sxx[1:, 1:] = (X_block.T @ X_block).astype(np.float64, copy=False)

    sy = float(y.sum(dtype=np.float64))
    Sxy[0] = sy
    Sxy[1:] = (X_block.T @ y).astype(np.float64, copy=False)

    Syy = float(y @ y)
    st = _ols_from_sufficient_stats(Sxx, Sxy, Syy, sy, int(n_block), q)

    info = {
        "n": int(n_block),
        "q": int(q),
        "k": int(k),
        "delta": float(delta),
        "block_cols": block_cols.tolist(),
        "coef_block": st["beta"][1:].copy(),
        "intercept": float(st["beta"][0]),
        "se_block": st["se"][1:].copy(),
        "t_block": st["t"][1:].copy(),
        "p_block": st["p"][1:].copy(),
        "intercept_p": float(st["p"][0]) if np.isfinite(st["p"][0]) else np.nan,
        "r2": float(st["r2"]),
        "aic": float(st["aic"]),
        "bic": float(st["bic"]),
        "df_resid": int(st["df_resid"]),
        "rng_seed": int(rng_seed),
    }
    return info


def oof_proba_per_cluster(X_all_df, y_all, c_all, n_splits=5, random_state=42):
    """
    以分群為單位做 StratifiedKFold OOF，回傳每筆真實點的 OOF 機率 p_oof。
    每個 fold 的 training 內部再做 SMOTE，避免合成點寫回 OOF。
    """
    y_all = np.asarray(y_all).astype(int)
    p_oof = np.full(len(y_all), np.nan, dtype=float)
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    for c in [0, 1]:
        idx = np.where(c_all == c)[0]
        Xc = X_all_df.iloc[idx].values
        yc = y_all[idx]

        for tr, te in skf.split(Xc, yc):
            X_tr, y_tr = Xc[tr], yc[tr]
            X_te = Xc[te]

            sm = SMOTE(sampling_strategy=1, k_neighbors=5, random_state=random_state)
            X_tr, y_tr = sm.fit_resample(X_tr, y_tr)

            m = XGBClassifier(
                n_estimators=300,
                max_depth=3,
                learning_rate=0.1,
                subsample=0.8,
                colsample_bytree=0.8,
                min_child_weight=10,
                gamma=1.0,
                reg_lambda=2.0,
                reg_alpha=0.5,
                random_state=random_state,
                n_jobs=-1,
                objective="binary:logistic",
                eval_metric="logloss",
                tree_method="hist",
            )
            m.fit(X_tr, y_tr)
            p_oof[idx[te]] = m.predict_proba(X_te)[:, 1]

    return p_oof

def pick_x0_indices(p, target, topk, min_p=None):
    """
    依 |p - target| 選出最接近 target 的 topk 索引（可加 min_p 過濾）。
    """
    p = np.asarray(p).reshape(-1)
    idx = np.arange(len(p))
    if min_p is not None:
        idx = idx[p[idx] >= min_p]
    if idx.size == 0:
        return idx
    order = np.argsort(np.abs(p[idx] - target))
    return idx[order[:min(topk, idx.size)]]


def run_Nround_fullp_screening(
    x0,
    blackbox_model,
    feature_names,
    thresholds,
    k=3,
    delta=0.1,
    scaler=None,
    n_block=800_000,
    base_seed=42,
):
    """
    對單一 (x0, blackbox) 做多輪抽樣 OLS：
    每輪抽 n_block 個格點、用 active_cols 做 OLS，依門檻縮小 active_cols。
    回傳每輪的統計表 df_all 與最終保留欄位 final_cols。
    """
    p = len(feature_names)
    active_cols = np.arange(p, dtype=np.int64)
    all_round_dfs = []

    for r, thr in enumerate(thresholds, start=1):
        if active_cols.size == 0:
            break

        seed = int(base_seed + r * 100000)

        info = local_surrogate_block_sample_ols(
            x0=x0,
            blackbox_model=blackbox_model,
            block_cols=active_cols,
            delta=delta,
            k=k,
            scaler=scaler,
            n_block=n_block,
            rng_seed=seed,
            store_yhat=False,
            store_yblackbox=False,
        )

        rows = []
        for t, col in enumerate(active_cols):
            rows.append({
                "feature": feature_names[int(col)],
                "feature_idx": int(col),
                "coef": float(info["coef_block"][t]),
                "abs_coef": float(abs(info["coef_block"][t])),
                "t": float(info["t_block"][t]),
                "p": float(info["p_block"][t]),
                "r2_model": float(info["r2"]),
                "df_resid": int(info["df_resid"]),
                "rng_seed": int(info["rng_seed"]),
                "n_block": int(info["n"]),
                "round_id": int(r),
                "threshold": float(thr),
                "q_used": int(info["q"]),
            })

        df_r = pd.DataFrame(rows)
        if len(df_r):
            df_r = df_r.sort_values(["p", "abs_coef"], ascending=[True, False]).reset_index(drop=True)
            df_r["rank_p"] = np.arange(1, len(df_r) + 1)

        all_round_dfs.append(df_r)

        pass_cols = df_r.loc[df_r["p"] <= thr, "feature_idx"].unique()
        active_cols = np.asarray(pass_cols, dtype=np.int64)

    df_all = pd.concat(all_round_dfs, ignore_index=True) if len(all_round_dfs) else pd.DataFrame()
    return df_all, active_cols

In [5]:
secom = pd.read_csv(data_path, sep='\t')
outlier = X_pca_df[(X_pca_df['pc1']>10)|(X_pca_df['pc2']>10)].index.tolist()
secom = secom.drop(index=outlier).reset_index(drop=True)
print(f'PCA極端值有:{outlier}, \n共{len(outlier)}個')
X = secom.drop(columns=['Time', 'Pass/Fail'])
y = secom['Pass/Fail'].replace({-1:0, 1:1}).astype(int)

na_counts = X.isna().sum()
X = X.loc[:, na_counts <= 1000].copy()
uniq_counts = X.nunique(dropna=False)
X = X.loc[:, uniq_counts >= 10].copy()
X = X.loc[:, ~X.T.duplicated(keep='first')].copy()

imputer = KNNImputer(n_neighbors=10, weights='distance')
X_imp = pd.DataFrame(imputer.fit_transform(X), columns=X.columns, index=X.index)

scaler = StandardScaler()
X_std = pd.DataFrame(scaler.fit_transform(X_imp), columns=X.columns, index=X.index)

smote = SMOTE(random_state=42, sampling_strategy=1, k_neighbors=5)
X_smote, y_smote = smote.fit_resample(X_std, y)

# X_smote, y_smote 已經是你前面做完 SMOTE 後的標準化空間資料
X_smote_df = pd.DataFrame(X_smote, columns=X_std.columns)

# 1) 在同一個 SMOTE 空間做 PCA + KMeans，得到 cluster
pca = PCA(n_components=0.9, random_state=42)
X_pca = pca.fit_transform(X_smote_df)
n_real = len(X_std)

# 必要斷言：確保前段就是原始點
assert np.allclose(X_smote_df.iloc[:n_real].values, X_std.values, atol=1e-7, rtol=1e-7)
assert np.array_equal(np.asarray(y_smote)[:n_real], np.asarray(y))

kmeans = KMeans(n_clusters=2, n_init=10, max_iter=300, random_state=42)
cluster = kmeans.fit_predict(X_pca)

# X_smote_df 目前包含所有特徵 + "cluster"
X_smote_df["cluster"] = cluster

# 確保 X_all_df 不含 cluster 欄
X_feat_df = X_smote_df.drop(columns=["cluster"]).copy()
# 只取真實點（不含 SMOTE 合成點）來做 local surrogate 與訓練每群 blackbox
X_all_df = X_feat_df.iloc[:n_real].copy() # 真實點特徵（標準化空間）
c_all = X_smote_df.loc[:n_real-1, "cluster"].to_numpy() # 真實點分群
y_real = np.asarray(y) # 真實 y

def oof_proba_per_cluster(X_all_df, y_all, c_all, n_splits=5, random_state=42):
    """
    以分群為單位做 StratifiedKFold OOF，回傳每筆真實點的 OOF 機率 p_oof。
    每個 fold 的 training 內部再做 SMOTE，避免合成點寫回 OOF。
    """
    y_all = np.asarray(y_all).astype(int)
    p_oof = np.full(len(y_all), np.nan, dtype=float)
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    for c in [0, 1]:
        idx = np.where(c_all == c)[0]
        Xc = X_all_df.iloc[idx].values
        yc = y_all[idx]

        for tr, te in skf.split(Xc, yc):
            X_tr, y_tr = Xc[tr], yc[tr]
            X_te = Xc[te]

            sm = SMOTE(sampling_strategy=1, k_neighbors=5, random_state=random_state)
            X_tr, y_tr = sm.fit_resample(X_tr, y_tr)

            m = XGBClassifier(
                n_estimators=300,
                max_depth=3,
                learning_rate=0.1,
                subsample=0.8,
                colsample_bytree=0.8,
                min_child_weight=10,
                gamma=1.0,
                reg_lambda=2.0,
                reg_alpha=0.5,
                random_state=random_state,
                n_jobs=-1,
                objective="binary:logistic",
                eval_metric="logloss",
                tree_method="hist",
            )
            m.fit(X_tr, y_tr)
            p_oof[idx[te]] = m.predict_proba(X_te)[:, 1]

    return p_oof

# 1) 先算每群 OOF 機率（用來挑 x0）
p_hat_oof = oof_proba_per_cluster(X_all_df, y_real, c_all)

PCA極端值有:[8, 10, 14, 15, 23, 24, 25, 33, 36, 44, 46, 47, 51, 54, 57, 58, 61, 72, 84, 91, 93, 96, 99, 143, 144, 163, 172, 183, 186, 195, 196, 275, 457, 466, 539, 634, 709, 800, 1397, 1427, 1458, 1489], 
共42個


c:\Users\No\anaconda3\envs\Laptop_20250527\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] 系統找不到指定的檔案。
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "c:\Users\No\anaconda3\envs\Laptop_20250527\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "c:\Users\No\anaconda3\envs\Laptop_20250527\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\No\anaconda3\envs\Laptop_20250527\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "c:\Users\No\anaconda3\envs\Laptop_20250527\Lib\subprocess.py", line 1538

# hat p = 0.5

## c=0, topk=12

In [6]:
def fit_blackbox_per_cluster(X_all_df, y_all, c_all, random_state=42):
    """
    以分群為單位訓練最終 blackbox（群內先 SMOTE 再 fit），回傳 dict{cluster: model}。
    """
    y_all = np.asarray(y_all)
    c=0
    idx = np.where(c_all == c)[0]
    Xc = X_all_df.iloc[idx].values
    yc = y_all[idx]

    sm = SMOTE(sampling_strategy=1, k_neighbors=5, random_state=random_state)
    Xc_sm, yc_sm = sm.fit_resample(Xc, yc)

    m = XGBClassifier(
            n_estimators=300,
            max_depth=3,
            learning_rate=0.1,
            subsample=0.8,
            colsample_bytree=0.8,
            min_child_weight=10,
            gamma=1.0,
            reg_lambda=2.0,
            reg_alpha=0.5,
            random_state=42,
            n_jobs=-1,
            objective="binary:logistic",
            eval_metric="logloss",
            tree_method="hist",
        )
    m.fit(Xc_sm, yc_sm)

    return {c: m}

# 2) 最後用全資料重訓每群最終黑箱（用於 lattice 探測與 surrogate 擬合）
blackbox_by_0 = fit_blackbox_per_cluster(X_all_df, y_real, c_all)

def run_block_pvalues_per_cluster_center_Nrounds_fullp(
    X_all_df, c_all, blackbox_by_c, p_hat_oof, thresholds,
    topk, target, k, n_block,
    delta=0.1, base_seed=42,
):
    """
    對每個 cluster：挑選 x0（靠近 target 機率），取中心點 x_center，
    再呼叫 run_3round_fullp_screening 得到 df 與每群的最終特徵集合。
    """
    out = []
    cfinal_col = {}
    feature_names = list(X_all_df.columns)

    c=0
    idx_c = np.where(c_all == c)[0]
    m = blackbox_by_c[c]
    p_hat_c = np.asarray(p_hat_oof)[idx_c]

    loc = pick_x0_indices(p_hat_c, target=target, topk=topk)
    if len(loc) == 0:
        cfinal_col[c] = np.array([], dtype=int)
        return pd.DataFrame(), cfinal_col

    global_ids = [int(idx_c[pos]) for pos in loc]
    x_center = X_all_df.iloc[global_ids].to_numpy().mean(axis=0)

    df_rounds, final_cols = run_Nround_fullp_screening(
        x0=x_center,
        blackbox_model=m,
        feature_names=feature_names,
        k=k,
        delta=delta,
        scaler=None,
        n_block=n_block,
        base_seed=base_seed + 1000 * c,
        thresholds=thresholds,
    )

    cfinal_col[c] = np.asarray(final_cols, dtype=int)

    if len(df_rounds):
        df_rounds["cluster"] = c
        df_rounds["x0_indices"] = [global_ids] * len(df_rounds)
        df_rounds["p_oof_center"] = float(p_hat_c[loc].mean())
        df_rounds["p_final_center"] = float(m.predict_proba(x_center.reshape(1, -1))[:, 1][0])
        out.append(df_rounds)

    df_all = pd.concat(out, ignore_index=True) if len(out) else pd.DataFrame()
    return df_all, cfinal_col


def run_repeat_screening_stats(
    X_all_df, c_all, blackbox_by_c, p_hat_oof,
    thresholds, topk, target, n_block,
    k=2, 
    B=50,
    delta=0.1,
    base_seed=42,
):
    """
    重複跑 B 次 N-round 篩選（每次用不同 seed），回傳：
      1) df_all_runs：所有 run 的 df_rounds 串起來（可選）
      2) df_summary：每群每特徵的選中次數與 beta(mean±SE)
    """
    feature_names = list(X_all_df.columns)
    p = len(feature_names)
    c=0
    # 收集：每個 cluster、每個 feature 的「被選到 p* 次數」與「最終輪 beta 列表」
    select_count = {c: np.zeros(p, dtype=np.int32)}
    beta_list = {c: {i: [] for i in range(p)}}

    df_all_runs = []

    for b in range(B):
        # 每次重複都換一組 seed（確保每輪抽到不同 D、跨 run 也不同）
        run_seed = int(base_seed + 1_000_000 * b)

        df_all_p, cfinal_col = run_block_pvalues_per_cluster_center_Nrounds_fullp(
            X_all_df=X_all_df,
            c_all=c_all,
            blackbox_by_c=blackbox_by_c,
            p_hat_oof=p_hat_oof,
            thresholds=thresholds,
            topk=topk,
            target=target,
            k=k,
            delta=delta,
            n_block=n_block,
            base_seed=run_seed,
        )

        if len(df_all_p):
            df_all_p = df_all_p.copy()
            df_all_p["run_id"] = b
            df_all_runs.append(df_all_p)

        # 逐 cluster：統計 p* 次數與最終輪 beta
        final_cols = np.asarray(cfinal_col.get(c, []), dtype=int)
        if final_cols.size == 0:
            continue

        select_count[c][final_cols] += 1

        # 從 df_all_p 取該 cluster 的「最後一輪」結果，抓出 p* 的 coef
        df_c = df_all_p[df_all_p["cluster"] == c]
        if len(df_c) == 0:
            continue

        last_round = int(df_c["round_id"].max())
        df_last = df_c[df_c["round_id"] == last_round]

        df_last_star = df_last[df_last["feature_idx"].isin(final_cols)][["feature_idx", "coef"]]
        for _, row in df_last_star.iterrows():
            beta_list[c][int(row["feature_idx"])].append(float(row["coef"]))

    # 彙總成表
    rows = []
    cnt = select_count[c]
    for j in np.where(cnt > 0)[0]:
        betas = np.asarray(beta_list[c][int(j)], dtype=float)
        if betas.size == 0:
            continue
        mean = float(betas.mean())
        se = float(betas.std(ddof=1) / np.sqrt(betas.size)) if betas.size >= 2 else np.nan

        rows.append({
            "cluster": c,
            "feature": feature_names[int(j)],
            "feature_idx": int(j),
            "select_count": int(cnt[int(j)]),
            "select_rate": float(cnt[int(j)] / B),
            "beta_n": int(betas.size),
            "beta_mean": mean,
            "beta_se": se,
            "beta_mean_minus_se": float(mean - se) if np.isfinite(se) else np.nan,
            "beta_mean_plus_se": float(mean + se) if np.isfinite(se) else np.nan,
        })

    df_summary = pd.DataFrame(rows)
    if len(df_summary):
        df_summary = df_summary.sort_values(
            ["cluster", "select_count", "beta_mean"],
            ascending=[True, False, False]
        ).reset_index(drop=True)

    df_all_runs = pd.concat(df_all_runs, ignore_index=True) if len(df_all_runs) else pd.DataFrame()
    return df_all_runs, df_summary

thresholds = (0.01, 0.005, 0.001, 0.0005, 0.0001)

# ====== 呼叫：跑 5 輪、重複 50 次，輸出統計 ======
df_all_runs, df_summary = run_repeat_screening_stats(
    X_all_df=X_all_df,
    c_all=c_all,
    blackbox_by_c=blackbox_by_0,
    p_hat_oof=p_hat_oof,
    thresholds=thresholds,
    B=50,
    topk=12,
    target=0.5,
    k=2,
    delta=0.1,
    n_block=1_000_000,
    base_seed=42,
)

# df_summary：每群每特徵的選中次數與 beta(mean±SE)
# df_all_runs：保留所有 run 的逐輪結果（可用來畫分佈或檢查每輪縮減情形）

In [7]:
df_summary.to_csv('p=half_c0_summary_B=50_topk=12_k=2_delta=01_n=100w.csv', sep='\t')

In [8]:
df_all_runs.to_csv('p=half_c0_allruns_B=50_topk=12_k=2_delta=01_n=100w.csv', sep='\t')

## c=1, topk=7

In [14]:
def fit_blackbox_per_cluster(X_all_df, y_all, c_all, random_state=42):
    """
    以分群為單位訓練最終 blackbox（群內先 SMOTE 再 fit），回傳 dict{cluster: model}。
    """
    y_all = np.asarray(y_all)
    
    c=1
    idx = np.where(c_all == c)[0]
    Xc = X_all_df.iloc[idx].values
    yc = y_all[idx]

    sm = SMOTE(sampling_strategy=1, k_neighbors=5, random_state=random_state)
    Xc_sm, yc_sm = sm.fit_resample(Xc, yc)

    m = XGBClassifier(
            n_estimators=300,
            max_depth=3,
            learning_rate=0.1,
            subsample=0.8,
            colsample_bytree=0.8,
            min_child_weight=10,
            gamma=1.0,
            reg_lambda=2.0,
            reg_alpha=0.5,
            random_state=42,
            n_jobs=-1,
            objective="binary:logistic",
            eval_metric="logloss",
            tree_method="hist",
        )
    m.fit(Xc_sm, yc_sm)

    return {c: m}

# 2) 最後用全資料重訓每群最終黑箱（用於 lattice 探測與 surrogate 擬合）
blackbox_by_1 = fit_blackbox_per_cluster(X_all_df, y_real, c_all)

def run_block_pvalues_per_cluster_center_Nrounds_fullp(
    X_all_df, c_all, blackbox_by_c, p_hat_oof, thresholds,
    topk, target, k, n_block,
    delta=0.1, base_seed=42,
):
    """
    對每個 cluster：挑選 x0（靠近 target 機率），取中心點 x_center，
    再呼叫 run_3round_fullp_screening 得到 df 與每群的最終特徵集合。
    """
    out = []
    cfinal_col = {}
    feature_names = list(X_all_df.columns)

    c=1
    idx_c = np.where(c_all == c)[0]
    m = blackbox_by_c[c]
    p_hat_c = np.asarray(p_hat_oof)[idx_c]

    loc = pick_x0_indices(p_hat_c, target=target, topk=topk)
    if len(loc) == 0:
        cfinal_col[c] = np.array([], dtype=int)
        return pd.DataFrame(), cfinal_col

    global_ids = [int(idx_c[pos]) for pos in loc]
    x_center = X_all_df.iloc[global_ids].to_numpy().mean(axis=0)

    df_rounds, final_cols = run_Nround_fullp_screening(
        x0=x_center,
        blackbox_model=m,
        feature_names=feature_names,
        k=k,
        delta=delta,
        scaler=None,
        n_block=n_block,
        base_seed=base_seed + 1000 * c,
        thresholds=thresholds,
    )

    cfinal_col[c] = np.asarray(final_cols, dtype=int)

    if len(df_rounds):
        df_rounds["cluster"] = c
        df_rounds["x0_indices"] = [global_ids] * len(df_rounds)
        df_rounds["p_oof_center"] = float(p_hat_c[loc].mean())
        df_rounds["p_final_center"] = float(m.predict_proba(x_center.reshape(1, -1))[:, 1][0])
        out.append(df_rounds)

    df_all = pd.concat(out, ignore_index=True) if len(out) else pd.DataFrame()
    return df_all, cfinal_col


def run_repeat_screening_stats(
    X_all_df, c_all, blackbox_by_c, p_hat_oof,
    thresholds, topk, target, n_block,
    k=2, 
    B=50,
    delta=0.1,
    base_seed=42,
):
    """
    重複跑 B 次 N-round 篩選（每次用不同 seed），回傳：
      1) df_all_runs：所有 run 的 df_rounds 串起來（可選）
      2) df_summary：每群每特徵的選中次數與 beta(mean±SE)
    """
    feature_names = list(X_all_df.columns)
    p = len(feature_names)
    c=1
    # 收集：每個 cluster、每個 feature 的「被選到 p* 次數」與「最終輪 beta 列表」
    select_count = {c: np.zeros(p, dtype=np.int32)}
    beta_list = {c: {i: [] for i in range(p)}}

    df_all_runs = []

    for b in range(B):
        # 每次重複都換一組 seed（確保每輪抽到不同 D、跨 run 也不同）
        run_seed = int(base_seed + 1_000_000 * b)

        df_all_p, cfinal_col = run_block_pvalues_per_cluster_center_Nrounds_fullp(
            X_all_df=X_all_df,
            c_all=c_all,
            blackbox_by_c=blackbox_by_c,
            p_hat_oof=p_hat_oof,
            thresholds=thresholds,
            topk=topk,
            target=target,
            k=k,
            delta=delta,
            n_block=n_block,
            base_seed=run_seed,
        )

        if len(df_all_p):
            df_all_p = df_all_p.copy()
            df_all_p["run_id"] = b
            df_all_runs.append(df_all_p)

        # 逐 cluster：統計 p* 次數與最終輪 beta
        final_cols = np.asarray(cfinal_col.get(c, []), dtype=int)
        if final_cols.size == 0:
            continue

        select_count[c][final_cols] += 1

        # 從 df_all_p 取該 cluster 的「最後一輪」結果，抓出 p* 的 coef
        df_c = df_all_p[df_all_p["cluster"] == c]
        if len(df_c) == 0:
            continue

        last_round = int(df_c["round_id"].max())
        df_last = df_c[df_c["round_id"] == last_round]

        df_last_star = df_last[df_last["feature_idx"].isin(final_cols)][["feature_idx", "coef"]]
        for _, row in df_last_star.iterrows():
            beta_list[c][int(row["feature_idx"])].append(float(row["coef"]))

    # 彙總成表
    rows = []
    cnt = select_count[c]
    for j in np.where(cnt > 0)[0]:
        betas = np.asarray(beta_list[c][int(j)], dtype=float)
        if betas.size == 0:
            continue
        mean = float(betas.mean())
        se = float(betas.std(ddof=1) / np.sqrt(betas.size)) if betas.size >= 2 else np.nan

        rows.append({
            "cluster": c,
            "feature": feature_names[int(j)],
            "feature_idx": int(j),
            "select_count": int(cnt[int(j)]),
            "select_rate": float(cnt[int(j)] / B),
            "beta_n": int(betas.size),
            "beta_mean": mean,
            "beta_se": se,
            "beta_mean_minus_se": float(mean - se) if np.isfinite(se) else np.nan,
            "beta_mean_plus_se": float(mean + se) if np.isfinite(se) else np.nan,
        })

    df_summary = pd.DataFrame(rows)
    if len(df_summary):
        df_summary = df_summary.sort_values(
            ["cluster", "select_count", "beta_mean"],
            ascending=[True, False, False]
        ).reset_index(drop=True)

    df_all_runs = pd.concat(df_all_runs, ignore_index=True) if len(df_all_runs) else pd.DataFrame()
    return df_all_runs, df_summary

thresholds = (0.01, 0.005, 0.001, 0.0005, 0.0001)

# ====== 呼叫：跑 5 輪、重複 50 次，輸出統計 ======
df1_all_runs, df1_summary = run_repeat_screening_stats(
    X_all_df=X_all_df,
    c_all=c_all,
    blackbox_by_c=blackbox_by_1,
    p_hat_oof=p_hat_oof,
    thresholds=thresholds,
    B=50,
    topk=7,
    target=0.5,
    k=2,
    delta=0.1,
    n_block=1_000_000,
    base_seed=42,
)

# df_summary：每群每特徵的選中次數與 beta(mean±SE)
# df_all_runs：保留所有 run 的逐輪結果（可用來畫分佈或檢查每輪縮減情形）

In [15]:
df1_summary.to_csv('p=half_c1_summary_B=50_topk=7_k=2_delta=01_n=100w.csv', sep='\t')

In [16]:
df1_all_runs.to_csv('p=half_c1_allruns_B=50_topk=7_k=2_delta=01_n=100w.csv', sep='\t')

---
---

In [9]:
# ===== (A) 最小新增：用 final features 在 x0 周圍抽大量格點、分批累積 sufficient stats =====

def local_surrogate_fixed_features_bigN(
    x0,
    blackbox_model,
    block_cols,          # final features indices
    delta,
    k,
    scaler=None,
    n_total=2_000_000,
    rng_seed=42,
    center_first=True,
):
    """
    固定特徵集合 block_cols (=final features)，在 x0 周圍一次抽 n_total 格點，
    以黑箱輸出 y_hat 做目標，對 X_block (=delta*offsets) 做 OLS：
        y_hat ~ [1, X_block]
    """
    x0 = np.asarray(x0, np.float32).reshape(1, -1)

    if scaler is not None:
        x0_scaled = scaler.transform(x0).astype(np.float32, copy=False)
    else:
        x0_scaled = x0.copy()

    p_model = x0_scaled.shape[1]
    block_cols = np.asarray(block_cols, dtype=np.int64)
    q = int(block_cols.size)
    if q < 1:
        raise ValueError("block_cols 至少 1 維")
    if k < 0:
        raise ValueError("k must be >= 0")
    if n_total < 1:
        raise ValueError("n_total must be >= 1")

    rng = np.random.default_rng(int(rng_seed))

    # 1) 一次生成 offsets / X_block（只在 q 維）
    offsets_q = _sample_offsets_block(
        q, k, int(n_total),
        center_first=bool(center_first),
        rng=rng
    )  # (n_total, q) float32
    X_block = (np.float32(delta) * offsets_q).astype(np.float32, copy=False)

    # 2) 一次組出完整 Z（n_total, p_model）再丟黑箱
    Zs = np.empty((int(n_total), p_model), dtype=np.float32)
    Zs[:] = x0_scaled
    Zs[:, block_cols] += X_block

    if scaler is not None:
        Z = scaler.inverse_transform(Zs).astype(np.float32, copy=False)
    else:
        Z = Zs

    if hasattr(blackbox_model, "predict_proba"):
        y = blackbox_model.predict_proba(Z)[:, 1]
    else:
        y = blackbox_model.predict(Z)

    y = np.asarray(y, dtype=np.float32).reshape(-1)
    if y.size != int(n_total):
        raise ValueError("blackbox 輸出長度與 n_total 不一致")
    
    Z_sub = Zs[:, block_cols].astype(np.float64, copy=False)

    # 3) sufficient stats（一次算完）
    Sxx = np.zeros((q + 1, q + 1), dtype=np.float64)
    Sxy = np.zeros((q + 1,), dtype=np.float64)

    Sxx[0, 0] = float(n_total)
    sx = Z_sub.sum(axis=0, dtype=np.float64)
    Sxx[0, 1:] = sx
    Sxx[1:, 0] = sx
    Sxx[1:, 1:] = (Z_sub.T @ Z_sub).astype(np.float64, copy=False)

    sy = float(y.sum(dtype=np.float64))
    Sxy[0] = sy
    Sxy[1:] = (Z_sub.T @ y).astype(np.float64, copy=False)

    Syy = float(y @ y)
    st = _ols_from_sufficient_stats(Sxx, Sxy, Syy, sy, int(n_total), q)

    info = {
        "n": int(n_total),
        "q": int(q),
        "k": int(k),
        "delta": float(delta),
        "block_cols": block_cols.tolist(),
        "intercept": float(st["beta"][0]),
        "coef_block": st["beta"][1:].copy(),
        "se_block": st["se"][1:].copy(),
        "t_block": st["t"][1:].copy(),
        "p_block": st["p"][1:].copy(),
        "intercept_p": float(st["p"][0]) if np.isfinite(st["p"][0]) else np.nan,
        "r2": float(st["r2"]),
        "aic": float(st["aic"]),
        "bic": float(st["bic"]),
        "df_resid": int(st["df_resid"]),
        "rng_seed": int(rng_seed),
    }
    
    return info

In [10]:
# ===== (B) 最小新增：重複抽樣取得 R^2 點估計與信賴區間 =====

def estimate_r2_ci(
    x0,
    blackbox_model,
    block_cols,
    delta,
    k,
    scaler=None,
    n_total=2_000_000,
    n_rep=30,
    base_seed=42,
    center_first=True,
    alpha=0.05,
):
    """
    以不同 rng_seed 重複呼叫 local_surrogate_fixed_features_bigN，
    用 Monte Carlo 重抽分布估計 R^2 的點估計與信賴區間。
    """
    infos = []
    r2_vals = []

    for b in range(int(n_rep)):
        info_b = local_surrogate_fixed_features_bigN(
            x0=x0,
            blackbox_model=blackbox_model,
            block_cols=block_cols,
            delta=delta,
            k=k,
            scaler=scaler,
            n_total=n_total,
            rng_seed=int(base_seed + b),
            center_first=center_first,
        )
        infos.append(info_b)
        r2_vals.append(float(info_b["r2"]))

    r2_vals = np.asarray(r2_vals, dtype=float)

    out = {
        "n_rep": int(n_rep),
        "r2_point": float(r2_vals.mean()),   # 點估計：重抽平均
        "r2_sd": float(r2_vals.std(ddof=1)) if len(r2_vals) > 1 else np.nan,
        "r2_ci_lower": float(np.quantile(r2_vals, alpha / 2)),
        "r2_ci_upper": float(np.quantile(r2_vals, 1 - alpha / 2)),
        "r2_all": r2_vals,
        "infos": infos,
    }
    return out

## C=0

In [11]:

# ===== (B) 最小用法：先拿 final features，再做 200 萬格點的局部回歸 =====
# 這段直接接在你現有流程後面跑

# 1) 用原本流程先得到 df_all_runs, df_summary（你已經有了）
# df_all_runs, df_summary = run_repeat_screening_stats(...)
df_summary = pd.read_csv(r'C:\Users\No\Documents\GitHub\psychic-spoon\0 Meeting\p=half_c0_summary_B=50_topk=12_k=2_delta=01_n=100w.csv', sep='\t')
# 2) 取 cluster=0 的 final features（用 select_rate=1 或 select_count 最大那批；這裡用 select_rate==1）
final_cols_c0 = df_summary.query("cluster==0 and select_rate==1.0")["feature_idx"].to_numpy(dtype=int)

# 3) 取 cluster=0 的中心點 x_center（沿用你原本挑 x0 的方式，最小改：直接重算一次 x_center）
#    注意：這裡用同一套 pick_x0_indices + x_center 定義，與前面一致
c = 0
idx_c = np.where(c_all == c)[0]
p_hat_c = np.asarray(p_hat_oof)[idx_c]
loc = pick_x0_indices(p_hat_c, target=0.5, topk=12)
global_ids = [int(idx_c[pos]) for pos in loc]
x_center_c0 = X_all_df.iloc[global_ids].to_numpy().mean(axis=0)

# 4) 用 cluster=0 的最終黑箱模型
m0 = next(iter(blackbox_by_0.values()))

# 5) 抽 200 萬格點做一次 OLS（分批累積）
info_big = local_surrogate_fixed_features_bigN(
    x0=x_center_c0,
    blackbox_model=m0,
    block_cols=final_cols_c0,
    delta=0.1,
    k=2,
    scaler=None,           # 你現在黑箱吃的是標準化空間
    n_total=2_000_000,
    rng_seed=42,          # 固定即可重現
    center_first=True,
)

# 5-1) 最小新增：重複抽樣估 R^2 點估計與 95% CI
r2_res = estimate_r2_ci(
    x0=x_center_c0,
    blackbox_model=m0,
    block_cols=final_cols_c0,
    delta=0.1,
    k=2,
    scaler=None,
    n_total=2_000_000,
    n_rep=30,          # 可先 20 或 30；太慢再降
    base_seed=42,
    center_first=True,
    alpha=0.05,
)

# 6) 整理成一張表（係數、p-value、含 intercept）
feature_names = list(X_all_df.columns)
rows = [{
    "feature": "(intercept)",
    "feature_idx": -1,
    "coef": float(info_big["intercept"]),
    "se": np.nan,
    "t": np.nan,
    "p": float(info_big["intercept_p"]),
    "r2": float(info_big["r2"])
}]
for t, col in enumerate(final_cols_c0):
    rows.append({
        "feature": feature_names[int(col)],
        "feature_idx": int(col),
        "coef": float(info_big["coef_block"][t]),
        "se": float(info_big["se_block"][t]),
        "t": float(info_big["t_block"][t]),
        "p": float(info_big["p_block"][t]),
        "r2": float(info_big["r2"]),
    })

df_local_final = pd.DataFrame(rows).sort_values(["p"], ascending=True).reset_index(drop=True)

# df_local_final 就是：在 x_center_c0 周圍、只用 final features、抽 200 萬格點擬合出的局部線性模型
# 線性函數（標準化空間位移版本）：
#   y_hat ≈ intercept + Σ coef_j * (x_j - x0_j)   （此處 x_j 指標準化座標；(x_j-x0_j) 就是 X_block）


In [12]:
df_r2_summary = pd.DataFrame([{
    "cluster": 0,
    "r2_single": float(info_big["r2"]),
    "r2_point": float(r2_res["r2_point"]),
    "r2_sd": float(r2_res["r2_sd"]),
    "r2_ci_lower": float(r2_res["r2_ci_lower"]),
    "r2_ci_upper": float(r2_res["r2_ci_upper"]),
    "n_rep": int(r2_res["n_rep"]),
    "n_total_each": int(info_big["n"]),
}])

df_r2_summary

,cluster,r2_single,r2_point,r2_sd,r2_ci_lower,r2_ci_upper,n_rep,n_total_each
0,0,0.584269,0.58416,0.000362,0.583563,0.584737,30,2000000


In [20]:
df_local_final['coef'].reindex(df_local_final['coef'].abs().sort_values(ascending=False).index).head(31)

0     1.005144
65    0.034033
39   -0.029912
15   -0.028171
64    0.022993
63    0.022274
62    0.019723
61    0.017796
21   -0.015820
22   -0.014731
59    0.014099
60    0.013929
58    0.013283
57    0.011796
56    0.010932
55    0.009463
23   -0.009319
54    0.009020
25   -0.008795
24   -0.008775
53    0.008565
52    0.008468
26   -0.008034
51    0.007764
27   -0.007745
29   -0.007529
37   -0.007520
50    0.007486
30   -0.006877
49    0.006598
48    0.006592
Name: coef, dtype: float64

In [13]:
df_local_final.reindex(df_local_final['coef'].abs().sort_values(ascending=False).index).head(31)

,feature,feature_idx,coef,se,t,p,r2
0,(intercept),-1,1.005144,NaN,NaN,0.0,0.584269
65,x291,238,0.034033,0.000052,654.915465,0.0,0.584269
39,x7,5,-0.029912,0.000052,-575.609920,0.0,0.584269
15,x141,131,-0.028171,0.000052,-542.348112,0.0,0.584269
64,x87,78,0.022993,0.000052,442.487881,0.0,0.584269
63,x217,192,0.022274,0.000052,428.575932,0.0,0.584269
62,x461,364,0.019723,0.000052,379.675086,0.0,0.584269
61,x20,17,0.017796,0.000052,342.653693,0.0,0.584269
21,x58,52,-0.015820,0.000052,-304.602755,0.0,0.584269
22,x1,0,-0.014731,0.000052,-283.708169,0.0,0.584269


## C=1

In [17]:
# 1) 用原本流程先得到 df_all_runs, df_summary（你已經有了）
# df_all_runs, df_summary = run_repeat_screening_stats(...)
df1_summary = pd.read_csv(r'C:\Users\No\Documents\GitHub\psychic-spoon\0 Meeting\p=half_c1_summary_B=50_topk=7_k=2_delta=01_n=100w.csv', sep='\t')
# 2) 取 cluster=0 的 final features（用 select_rate=1 或 select_count 最大那批；這裡用 select_rate==1）
final_cols_c1 = df1_summary.query("cluster==1 and select_rate==1.0")["feature_idx"].to_numpy(dtype=int)

# 3) 取 cluster=0 的中心點 x_center（沿用你原本挑 x0 的方式，最小改：直接重算一次 x_center）
#    注意：這裡用同一套 pick_x0_indices + x_center 定義，與前面一致
c = 1
idx_c = np.where(c_all == c)[0]
p_hat_c = np.asarray(p_hat_oof)[idx_c]
loc = pick_x0_indices(p_hat_c, target=0.5, topk=7)
global_ids = [int(idx_c[pos]) for pos in loc]
x_center_c1 = X_all_df.iloc[global_ids].to_numpy().mean(axis=0)

# 4) 用 cluster=1 的最終黑箱模型
m1 = next(iter(blackbox_by_1.values()))

# 5) 抽 200 萬格點做一次 OLS（分批累積）
info_big = local_surrogate_fixed_features_bigN(
    x0=x_center_c1,
    blackbox_model=m1,
    block_cols=final_cols_c1,
    delta=0.1,
    k=2,
    scaler=None,           # 你現在黑箱吃的是標準化空間
    n_total=2_000_000,
    rng_seed=42,          # 固定即可重現
    center_first=True,
)

# 5-1) 最小新增：重複抽樣估 R^2 點估計與 95% CI
r2_res = estimate_r2_ci(
    x0=x_center_c1,
    blackbox_model=m1,
    block_cols=final_cols_c1,
    delta=0.1,
    k=2,
    scaler=None,
    n_total=2_000_000,
    n_rep=30,          # 可先 20 或 30；太慢再降
    base_seed=42,
    center_first=True,
    alpha=0.05,
)

# 6) 整理成一張表（係數、p-value、含 intercept）
feature_names = list(X_all_df.columns)
rows = [{
    "feature": "(intercept)",
    "feature_idx": -1,
    "coef": float(info_big["intercept"]),
    "se": np.nan,
    "t": np.nan,
    "p": float(info_big["intercept_p"]),
    "r2": float(info_big["r2"])
}]
for t, col in enumerate(final_cols_c1):
    rows.append({
        "feature": feature_names[int(col)],
        "feature_idx": int(col),
        "coef": float(info_big["coef_block"][t]),
        "se": float(info_big["se_block"][t]),
        "t": float(info_big["t_block"][t]),
        "p": float(info_big["p_block"][t]),
        "r2": float(info_big["r2"]),
    })

df1_local_final = pd.DataFrame(rows).sort_values(["p"], ascending=True).reset_index(drop=True)

# df_local_final 就是：在 x_center_c0 周圍、只用 final features、抽 200 萬格點擬合出的局部線性模型
# 線性函數（標準化空間位移版本）：
#   y_hat ≈ intercept + Σ coef_j * (x_j - x0_j)   （此處 x_j 指標準化座標；(x_j-x0_j) 就是 X_block）


In [18]:
df_r2_summary = pd.DataFrame([{
    "cluster": 0,
    "r2_single": float(info_big["r2"]),
    "r2_point": float(r2_res["r2_point"]),
    "r2_sd": float(r2_res["r2_sd"]),
    "r2_ci_lower": float(r2_res["r2_ci_lower"]),
    "r2_ci_upper": float(r2_res["r2_ci_upper"]),
    "n_rep": int(r2_res["n_rep"]),
    "n_total_each": int(info_big["n"]),
}])

df_r2_summary

,cluster,r2_single,r2_point,r2_sd,r2_ci_lower,r2_ci_upper,n_rep,n_total_each
0,0,0.592629,0.592644,0.001072,0.590905,0.59456,30,2000000


In [19]:
df1_local_final.reindex(df1_local_final['coef'].abs().sort_values(ascending=False).index).head(31)

,feature,feature_idx,coef,se,t,p,r2
0,(intercept),-1,0.996643,NaN,NaN,0.0,0.592629
27,x386,306,0.016675,0.00002,838.248099,0.0,0.592629
50,x117,107,-0.012556,0.00002,-631.552533,0.0,0.592629
2,x420,328,-0.011799,0.00002,-593.573937,0.0,0.592629
28,x456,360,0.010252,0.00002,515.514920,0.0,0.592629
22,x28,25,-0.009172,0.00002,-461.531607,0.0,0.592629
21,x36,33,-0.008981,0.00002,-451.759581,0.0,0.592629
20,x287,234,-0.007539,0.00002,-378.928419,0.0,0.592629
19,x103,93,-0.006848,0.00002,-344.319662,0.0,0.592629
29,x583,451,0.005797,0.00002,291.682462,0.0,0.592629


---
---

In [ ]:
# ===== (A) 最小新增：用 final features 在 x0 周圍抽大量格點、分批累積 sufficient stats =====

def local_surrogate_fixed_features_bigN(
    x0,
    blackbox_model,
    block_cols,          # final features indices
    delta,
    k,
    scaler=None,
    n_total=2_000_000,
    rng_seed=42,
    center_first=True,
):
    """
    固定特徵集合 block_cols (=final features)，在 x0 周圍一次抽 n_total 格點，
    以黑箱輸出 y_hat 做目標，對 X_block (=delta*offsets) 做 OLS：
        y_hat ~ [1, X_block]
    """
    x0 = np.asarray(x0, np.float32).reshape(1, -1)

    if scaler is not None:
        x0_scaled = scaler.transform(x0).astype(np.float32, copy=False)
    else:
        x0_scaled = x0.copy()

    p_model = x0_scaled.shape[1]
    block_cols = np.asarray(block_cols, dtype=np.int64)
    q = int(block_cols.size)
    if q < 1:
        raise ValueError("block_cols 至少 1 維")
    if k < 0:
        raise ValueError("k must be >= 0")
    if n_total < 1:
        raise ValueError("n_total must be >= 1")

    rng = np.random.default_rng(int(rng_seed))

    # 1) 一次生成 offsets / X_block（只在 q 維）
    offsets_q = _sample_offsets_block(
        q, k, int(n_total),
        center_first=bool(center_first),
        rng=rng
    )  # (n_total, q) float32
    X_block = (np.float32(delta) * offsets_q).astype(np.float32, copy=False)

    # 2) 一次組出完整 Z（n_total, p_model）再丟黑箱
    Zs = np.empty((int(n_total), p_model), dtype=np.float32)
    Zs[:] = x0_scaled
    Zs[:, block_cols] += X_block

    if scaler is not None:
        Z = scaler.inverse_transform(Zs).astype(np.float32, copy=False)
    else:
        Z = Zs

    if hasattr(blackbox_model, "predict_proba"):
        y = blackbox_model.predict_proba(Z)[:, 1]
    else:
        y = blackbox_model.predict(Z)

    y = np.asarray(y, dtype=np.float32).reshape(-1)
    if y.size != int(n_total):
        raise ValueError("blackbox 輸出長度與 n_total 不一致")
    
    Z_sub = Zs[:, block_cols].astype(np.float64, copy=False)

    # 3) sufficient stats（一次算完）
    Sxx = np.zeros((q + 1, q + 1), dtype=np.float64)
    Sxy = np.zeros((q + 1,), dtype=np.float64)

    Sxx[0, 0] = float(n_total)
    sx = Z_sub.sum(axis=0, dtype=np.float64)
    Sxx[0, 1:] = sx
    Sxx[1:, 0] = sx
    Sxx[1:, 1:] = (Z_sub.T @ Z_sub).astype(np.float64, copy=False)

    sy = float(y.sum(dtype=np.float64))
    Sxy[0] = sy
    Sxy[1:] = (Z_sub.T @ y).astype(np.float64, copy=False)

    Syy = float(y @ y)
    st = _ols_from_sufficient_stats(Sxx, Sxy, Syy, sy, int(n_total), q)

    X_design = sm.add_constant(X_block, has_constant='add')
    ols_res = sm.OLS(y, X_design).fit()
    y_hat = ols_res.fittedvalues
    resid = ols_res.resid

    info = {
        "n": int(n_total),
        "q": int(q),
        "k": int(k),
        "delta": float(delta),
        "block_cols": block_cols.tolist(),
        "intercept": float(st["beta"][0]),
        "coef_block": st["beta"][1:].copy(),
        "se_block": st["se"][1:].copy(),
        "t_block": st["t"][1:].copy(),
        "p_block": st["p"][1:].copy(),
        "intercept_p": float(st["p"][0]) if np.isfinite(st["p"][0]) else np.nan,
        "r2": float(st["r2"]),
        "aic": float(st["aic"]),
        "bic": float(st["bic"]),
        "df_resid": int(st["df_resid"]),
        "rng_seed": int(rng_seed),
        "x0_center": x0_scaled.reshape(-1).copy(),   # 此局部模型所對應的中心點（標準化空間）
        "X_local_full": Zs.copy(),                   # 完整局部資料集（標準化空間）
        "X_block_shift": X_block.copy(),             # block_cols 上的相對位移
    }
    return info

def predict_local_surrogate_from_fullX(X_full, x0_center, block_cols, intercept, coef_block):
    """
    用某群的局部線性模型 g_c 預測任意完整特徵矩陣 X_full 上的值
    X_full 必須與 x0_center 在同一空間（你這裡都是標準化空間）
    """
    X_full = np.asarray(X_full, dtype=np.float64)
    x0_center = np.asarray(x0_center, dtype=np.float64).reshape(-1)
    block_cols = np.asarray(block_cols, dtype=int)

    X_shift = X_full[:, block_cols] - x0_center[block_cols]
    y_pred = float(intercept) + X_shift @ np.asarray(coef_block, dtype=np.float64)
    return np.asarray(y_pred, dtype=np.float64)

def eval_surrogate(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)

    return {
        "r2": float(r2_score(y_true, y_pred)),
        "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "corr": float(np.corrcoef(y_true, y_pred)[0, 1]),
    }

df_summary = pd.read_csv(r'C:\Users\No\Documents\GitHub\psychic-spoon\0 Meeting\p=half_c0_summary_B=50_topk=12_k=2_delta=01_n=100w.csv', sep='\t')
final_cols_c0 = df_summary.query("cluster==0 and select_rate==1.0")["feature_idx"].to_numpy(dtype=int)

# 3) 取 cluster=0 的中心點 x_center（沿用你原本挑 x0 的方式，最小改：直接重算一次 x_center）
#    注意：這裡用同一套 pick_x0_indices + x_center 定義，與前面一致
c = 0
idx_0 = np.where(c_all == c)[0]
p_hat_0 = np.asarray(p_hat_oof)[idx_0]
loc = pick_x0_indices(p_hat_0, target=0.5, topk=12)
global_ids = [int(idx_0[pos]) for pos in loc]
x_center_c0 = X_all_df.iloc[global_ids].to_numpy().mean(axis=0)

# 4) 用 cluster=0 的最終黑箱模型
m0 = next(iter(blackbox_by_0.values()))

info_big_c0 = local_surrogate_fixed_features_bigN(
    x0=x_center_c0,
    blackbox_model=m0,
    block_cols=final_cols_c0,
    delta=0.1,
    k=5,
    scaler=None,
    n_total=2_000_000,
    rng_seed=42,
    center_first=True,
)

df1_summary = pd.read_csv(r'C:\Users\No\Documents\GitHub\psychic-spoon\0 Meeting\p=half_c1_summary_B=50_topk=7_k=2_delta=01_n=100w.csv', sep='\t')
final_cols_c1 = df1_summary.query("cluster==1 and select_rate==1.0")["feature_idx"].to_numpy(dtype=int)

c = 1
idx_1 = np.where(c_all == c)[0]
p_hat_1 = np.asarray(p_hat_oof)[idx_1]
loc = pick_x0_indices(p_hat_1, target=0.5, topk=12)
global_ids = [int(idx_1[pos]) for pos in loc]
x_center_c1 = X_all_df.iloc[global_ids].to_numpy().mean(axis=0)

m1 = next(iter(blackbox_by_1.values()))

info_big_c1 = local_surrogate_fixed_features_bigN(
    x0=x_center_c1,
    blackbox_model=m1,
    block_cols=final_cols_c1,
    delta=0.1,
    k=5,
    scaler=None,
    n_total=2_000_000,
    rng_seed=42,
    center_first=True,
)

f0_on_G0 = info_big_c0["y_blackbox"]

g0_on_G0 = predict_local_surrogate_from_fullX(
    X_full=info_big_c0["X_local_full"],
    x0_center=info_big_c0["x0_center"],
    block_cols=info_big_c0["block_cols"],
    intercept=info_big_c0["intercept"],
    coef_block=info_big_c0["coef_block"],
)

eval_g0_in = eval_surrogate(f0_on_G0, g0_on_G0)

f1_on_G1 = info_big_c1["y_blackbox"]

g1_on_G1 = predict_local_surrogate_from_fullX(
    X_full=info_big_c1["X_local_full"],
    x0_center=info_big_c1["x0_center"],
    block_cols=info_big_c1["block_cols"],
    intercept=info_big_c1["intercept"],
    coef_block=info_big_c1["coef_block"],
)

eval_g1_in = eval_surrogate(f1_on_G1, g1_on_G1)

g0_on_G1 = predict_local_surrogate_from_fullX(
    X_full=info_big_c1["X_local_full"],      # 用 G1
    x0_center=info_big_c0["x0_center"],      # 仍用 g0 自己的中心點
    block_cols=info_big_c0["block_cols"],    # 仍用 g0 自己的特徵
    intercept=info_big_c0["intercept"],
    coef_block=info_big_c0["coef_block"],
)

eval_g0_to_G1 = eval_surrogate(f1_on_G1, g0_on_G1)

g0_on_G1 = predict_local_surrogate_from_fullX(
    X_full=info_big_c1["X_local_full"],      # 用 G1
    x0_center=info_big_c0["x0_center"],      # 仍用 g0 自己的中心點
    block_cols=info_big_c0["block_cols"],    # 仍用 g0 自己的特徵
    intercept=info_big_c0["intercept"],
    coef_block=info_big_c0["coef_block"],
)

eval_g0_to_G1 = eval_surrogate(f1_on_G1, g0_on_G1)

g1_on_G0 = predict_local_surrogate_from_fullX(
    X_full=info_big_c0["X_local_full"],      # 用 G0
    x0_center=info_big_c1["x0_center"],      # 仍用 g1 自己的中心點
    block_cols=info_big_c1["block_cols"],    # 仍用 g1 自己的特徵
    intercept=info_big_c1["intercept"],
    coef_block=info_big_c1["coef_block"],
)

eval_g1_to_G0 = eval_surrogate(f0_on_G0, g1_on_G0)

df_transfer_compare = pd.DataFrame([
    {
        "setting": "in-domain: g0 on G0 vs f0(G0)",
        **eval_g0_in
    },
    {
        "setting": "in-domain: g1 on G1 vs f1(G1)",
        **eval_g1_in
    },
    {
        "setting": "transfer: g0 on G1 vs f1(G1)",
        **eval_g0_to_G1
    },
    {
        "setting": "transfer: g1 on G0 vs f0(G0)",
        **eval_g1_to_G0
    },
])

print(df_transfer_compare)

def plot_blackbox_vs_local(y_blackbox, y_local, title, sample_n=10000, seed=123):
    y_blackbox = np.asarray(y_blackbox)
    y_local = np.asarray(y_local)

    rng = np.random.default_rng(seed)
    idx = rng.choice(len(y_blackbox), size=min(sample_n, len(y_blackbox)), replace=False)

    x = y_blackbox[idx]
    y = y_local[idx]

    lo = min(x.min(), y.min())
    hi = max(x.max(), y.max())

    plt.figure(figsize=(5, 5))
    plt.scatter(x, y, s=8, alpha=0.3)
    plt.plot([lo, hi], [lo, hi], linestyle='--')
    plt.xlabel("Black-box output")
    plt.ylabel("Local surrogate output")
    plt.title(title)
    plt.tight_layout()
    plt.show()

plot_blackbox_vs_local(f0_on_G0, g0_on_G0, "In-domain: f0(G0) vs g0(G0)", seed=1)
plot_blackbox_vs_local(f1_on_G1, g1_on_G1, "In-domain: f1(G1) vs g1(G1)", seed=2)
plot_blackbox_vs_local(f1_on_G1, g0_on_G1, "Transfer: f1(G1) vs g0(G1)", seed=3)
plot_blackbox_vs_local(f0_on_G0, g1_on_G0, "Transfer: f0(G0) vs g1(G0)", seed=4)

---
---
# PLOT

In [ ]:
# ===== (A) 最小新增：用 final features 在 x0 周圍抽大量格點、分批累積 sufficient stats =====

def local_surrogate_fixed_features_bigN(
    x0,
    blackbox_model,
    block_cols,          # final features indices
    delta,
    k,
    scaler=None,
    n_total=2_000_000,
    rng_seed=42,
    center_first=True,
    return_diagnostics=False, 
):
    """
    固定特徵集合 block_cols (=final features)，在 x0 周圍一次抽 n_total 格點，
    以黑箱輸出 y_hat 做目標，對 X_block (=delta*offsets) 做 OLS：
        y_hat ~ [1, X_block]
    """
    x0 = np.asarray(x0, np.float32).reshape(1, -1)

    if scaler is not None:
        x0_scaled = scaler.transform(x0).astype(np.float32, copy=False)
    else:
        x0_scaled = x0.copy()

    p_model = x0_scaled.shape[1]
    block_cols = np.asarray(block_cols, dtype=np.int64)
    q = int(block_cols.size)
    if q < 1:
        raise ValueError("block_cols 至少 1 維")
    if k < 0:
        raise ValueError("k must be >= 0")
    if n_total < 1:
        raise ValueError("n_total must be >= 1")

    rng = np.random.default_rng(int(rng_seed))

    # 1) 一次生成 offsets / X_block（只在 q 維）
    offsets_q = _sample_offsets_block(
        q, k, int(n_total),
        center_first=bool(center_first),
        rng=rng
    )  # (n_total, q) float32
    X_block = (np.float32(delta) * offsets_q).astype(np.float32, copy=False)

    # 2) 一次組出完整 Z（n_total, p_model）再丟黑箱
    Zs = np.empty((int(n_total), p_model), dtype=np.float32)
    Zs[:] = x0_scaled
    Zs[:, block_cols] += X_block

    if scaler is not None:
        Z = scaler.inverse_transform(Zs).astype(np.float32, copy=False)
    else:
        Z = Zs

    if hasattr(blackbox_model, "predict_proba"):
        y = blackbox_model.predict_proba(Z)[:, 1]
    else:
        y = blackbox_model.predict(Z)

    y = np.asarray(y, dtype=np.float32).reshape(-1)
    if y.size != int(n_total):
        raise ValueError("blackbox 輸出長度與 n_total 不一致")
    
    Z_sub = Zs[:, block_cols].astype(np.float64, copy=False)

    # 3) sufficient stats（一次算完）
    Sxx = np.zeros((q + 1, q + 1), dtype=np.float64)
    Sxy = np.zeros((q + 1,), dtype=np.float64)

    Sxx[0, 0] = float(n_total)
    sx = Z_sub.sum(axis=0, dtype=np.float64)
    Sxx[0, 1:] = sx
    Sxx[1:, 0] = sx
    Sxx[1:, 1:] = (Z_sub.T @ Z_sub).astype(np.float64, copy=False)

    sy = float(y.sum(dtype=np.float64))
    Sxy[0] = sy
    Sxy[1:] = (Z_sub.T @ y).astype(np.float64, copy=False)

    Syy = float(y @ y)
    st = _ols_from_sufficient_stats(Sxx, Sxy, Syy, sy, int(n_total), q)

    if return_diagnostics:
        X_design = sm.add_constant(X_block, has_constant='add')
        ols_res = sm.OLS(y, X_design).fit()
        y_hat = ols_res.fittedvalues
        resid = ols_res.resid
    else:
        X_design = None
        ols_res = None
        y_hat = None
        resid = None

    info = {
        "n": int(n_total),
        "q": int(q),
        "k": int(k),
        "delta": float(delta),
        "block_cols": block_cols.tolist(),
        "intercept": float(st["beta"][0]),
        "coef_block": st["beta"][1:].copy(),
        "se_block": st["se"][1:].copy(),
        "t_block": st["t"][1:].copy(),
        "p_block": st["p"][1:].copy(),
        "intercept_p": float(st["p"][0]) if np.isfinite(st["p"][0]) else np.nan,
        "r2": float(st["r2"]),
        "aic": float(st["aic"]),
        "bic": float(st["bic"]),
        "df_resid": int(st["df_resid"]),
        "rng_seed": int(rng_seed),
    }

    if return_diagnostics:
        info.update({
            "x0_center": x0_scaled.reshape(-1).copy(),
            "X_local_full": Zs.copy(),
            "X_block_shift": X_block.copy(),
            "X_design": X_design,
            "y_blackbox": y.copy(),
            "y_fitted": np.asarray(y_hat).copy(),
            "resid": np.asarray(resid).copy(),
            "ols_result": ols_res,
        })
        
    return info

## C=0 QQPLOT RESIDUALPLOT

In [ ]:
df_summary = pd.read_csv(r'C:\Users\No\Documents\GitHub\psychic-spoon\0 Meeting\p=half_c0_summary_B=50_topk=12_k=2_delta=01_n=100w.csv', sep='\t')
final_cols_c0 = df_summary.query("cluster==0 and select_rate==1.0")["feature_idx"].to_numpy(dtype=int)

c = 0
idx_c = np.where(c_all == c)[0]
p_hat_c = np.asarray(p_hat_oof)[idx_c]
loc = pick_x0_indices(p_hat_c, target=0.5, topk=12)
global_ids = [int(idx_c[pos]) for pos in loc]
x_center_c0 = X_all_df.iloc[global_ids].to_numpy().mean(axis=0)

m0 = next(iter(blackbox_by_0.values()))

info_big = local_surrogate_fixed_features_bigN(
    x0=x_center_c0,
    blackbox_model=m0,
    block_cols=final_cols_c0,
    delta=0.1,
    k=5,
    scaler=None,           # 你現在黑箱吃的是標準化空間
    n_total=3_000,
    rng_seed=42,          # 固定即可重現
    center_first=True,
    return_diagnostics=True,   # 關鍵
)

# ===== (C) 單次模型診斷 =====
ols_res = info_big["ols_result"]
X_design = info_big["X_design"]
y_bb = np.asarray(info_big["y_blackbox"])
y_fit = np.asarray(info_big["y_fitted"])
resid = np.asarray(info_big["resid"])


# 1) Residual plot
plt.figure(figsize=(6, 4))
plt.scatter(y_fit, resid, s=8, alpha=0.3)
plt.axhline(0, color='red', linestyle='--', linewidth=1)
plt.xlabel("Fitted values")
plt.ylabel("Residuals")
plt.title("Residual Plot")
plt.tight_layout()
plt.show()

# 2) QQ plot
plt.figure(figsize=(6, 4))
sm.qqplot(resid, line='45', fit=True)
plt.title("Q-Q Plot of Residuals")
plt.tight_layout()
plt.show()

# 3) 統計檢定
dw_stat = durbin_watson(resid)

jb_stat, jb_pvalue, skew, kurtosis = jarque_bera(resid)

reset_res = linear_reset(ols_res, power=2, use_f=True)   # Ramsey RESET
rainbow_stat, rainbow_pvalue = linear_rainbow(ols_res)   # Rainbow test

bp_stat, bp_pvalue, f_stat, f_pvalue = het_breuschpagan(resid, X_design)

diag_table = pd.DataFrame({
    "test": [
        "Durbin-Watson",
        "Jarque-Bera",
        "Ramsey RESET (F)",
        "Rainbow",
        "Breusch-Pagan LM",
        "Breusch-Pagan F"
    ],
    "stat": [
        dw_stat,
        jb_stat,
        float(reset_res.fvalue),
        rainbow_stat,
        bp_stat,
        f_stat
    ],
    "p_value": [
        np.nan,                 # DW 通常不直接報 p-value
        jb_pvalue,
        float(reset_res.pvalue),
        rainbow_pvalue,
        bp_pvalue,
        f_pvalue
    ]
})

print(diag_table)

---
---

# hat p = 1

In [ ]:
secom = pd.read_csv(data_path, sep='\t')
outlier = X_pca_df[(X_pca_df['pc1']>10)|(X_pca_df['pc2']>10)].index.tolist()
secom = secom.drop(index=outlier).reset_index(drop=True)
print(f'PCA極端值有:{outlier}, \n共{len(outlier)}個')
X = secom.drop(columns=['Time', 'Pass/Fail'])
y = secom['Pass/Fail'].replace({-1:0, 1:1}).astype(int)

na_counts = X.isna().sum()
X = X.loc[:, na_counts <= 1000].copy()
uniq_counts = X.nunique(dropna=False)
X = X.loc[:, uniq_counts >= 10].copy()
X = X.loc[:, ~X.T.duplicated(keep='first')].copy()

imputer = KNNImputer(n_neighbors=10, weights='distance')
X_imp = pd.DataFrame(imputer.fit_transform(X), columns=X.columns, index=X.index)

scaler = StandardScaler()
X_std = pd.DataFrame(scaler.fit_transform(X_imp), columns=X.columns, index=X.index)

smote = SMOTE(random_state=42, sampling_strategy=1, k_neighbors=5)
X_smote, y_smote = smote.fit_resample(X_std, y)

# X_smote, y_smote 已經是你前面做完 SMOTE 後的標準化空間資料
X_smote_df = pd.DataFrame(X_smote, columns=X_std.columns)

# 1) 在同一個 SMOTE 空間做 PCA + KMeans，得到 cluster
pca = PCA(n_components=0.9, random_state=42)
X_pca = pca.fit_transform(X_smote_df)
n_real = len(X_std)

# 必要斷言：確保前段就是原始點
assert np.allclose(X_smote_df.iloc[:n_real].values, X_std.values, atol=1e-7, rtol=1e-7)
assert np.array_equal(np.asarray(y_smote)[:n_real], np.asarray(y))

kmeans = KMeans(n_clusters=2, n_init=10, max_iter=300, random_state=42)
cluster = kmeans.fit_predict(X_pca)

# X_smote_df 目前包含所有特徵 + "cluster"
X_smote_df["cluster"] = cluster

# 確保 X_all_df 不含 cluster 欄
X_feat_df = X_smote_df.drop(columns=["cluster"]).copy()
# 只取真實點（不含 SMOTE 合成點）來做 local surrogate 與訓練每群 blackbox
X_all_df = X_feat_df.iloc[:n_real].copy() # 真實點特徵（標準化空間）
c_all = X_smote_df.loc[:n_real-1, "cluster"].to_numpy() # 真實點分群
y_real = np.asarray(y) # 真實 y

# 1) 先算每群 OOF 機率（用來挑 x0）
p_hat_oof = oof_proba_per_cluster(X_all_df, y_real, c_all)

# 2) 最後用全資料重訓每群最終黑箱（用於 lattice 探測與 surrogate 擬合）
blackbox_by_c = fit_blackbox_per_cluster(X_all_df, y_real, c_all)


thresholds = (0.01, 0.005, 0.001, 0.0005, 0.0001)

# ====== 呼叫：跑 5 輪、重複 50 次，輸出統計 ======
df1_all_runs, df1_summary = run_repeat_screening_stats(
    X_all_df=X_all_df,
    c_all=c_all,
    blackbox_by_c=blackbox_by_c,
    p_hat_oof=p_hat_oof,
    thresholds=thresholds,
    B=50,
    topk=2,
    target=1,
    k=2,
    delta=0.1,
    n_block=800_000,
    base_seed=42,
)

# df_summary：每群每特徵的選中次數與 beta(mean±SE)
# df_all_runs：保留所有 run 的逐輪結果（可用來畫分佈或檢查每輪縮減情形）

In [ ]:
df1_summary[df1_summary['cluster']==0]

In [ ]:
df1_all_runs.head()

## C=0

In [ ]:
# ===== 先拿 final features，再做 200 萬格點的局部回歸 =====

# 1) 用原本流程先得到 df_all_runs, df_summary（你已經有了）
# df_all_runs, df_summary = run_repeat_screening_stats(...)

# 2) 取 cluster=0 的 final features（用 select_rate=1）
final1_cols_c0 = df1_summary.query("cluster==0 and select_rate==1.0")["feature_idx"].to_numpy(dtype=int)

# 3) 取 cluster=0 的中心點 x_center（沿用你原本挑 x0 的方式，最小改：直接重算一次 x_center）
#    注意：這裡用同一套 pick_x0_indices + x_center 定義，與前面一致
c = 0
idx_c = np.where(c_all == c)[0]
p_hat_c = np.asarray(p_hat_oof)[idx_c]
loc = pick_x0_indices(p_hat_c, target=1, topk=2)
global_ids = [int(idx_c[pos]) for pos in loc]
x_center_c0 = X_all_df.iloc[global_ids].to_numpy().mean(axis=0)

# 4) 用 cluster=0 的最終黑箱模型
m0 = blackbox_by_c[0]

# 5) 抽 200 萬格點做一次 OLS（分批累積）
info_big = local_surrogate_fixed_features_bigN(
    x0=x_center_c0,
    blackbox_model=m0,
    block_cols=final1_cols_c0,
    delta=0.1,
    k=2,
    scaler=None,           # 你現在黑箱吃的是標準化空間
    n_total=2_000_000,
    rng_seed=42,          # 固定即可重現
    center_first=True,
)

# 6) 整理成一張表（係數、p-value、含 intercept）
feature_names = list(X_all_df.columns)
rows = [{
    "feature": "(intercept)",
    "feature_idx": -1,
    "coef": float(info_big["intercept"]),
    "se": np.nan,
    "t": np.nan,
    "p": float(info_big["intercept_p"]),
    "r2": float(info_big["r2"])
}]
for t, col in enumerate(final1_cols_c0):
    rows.append({
        "feature": feature_names[int(col)],
        "feature_idx": int(col),
        "coef": float(info_big["coef_block"][t]),
        "se": float(info_big["se_block"][t]),
        "t": float(info_big["t_block"][t]),
        "p": float(info_big["p_block"][t]),
        "r2": float(info_big["r2"]),
    })

df1_local_final = pd.DataFrame(rows).sort_values(["p"], ascending=True).reset_index(drop=True)

# df_local_final 就是：在 x_center_c0 周圍、只用 final features、抽 200 萬格點擬合出的局部線性模型
# y_hat ≈ intercept + Σ coef_j * (x_j - x0_j)   （此處 x_j 指標準化座標；(x_j-x0_j) 就是 X_block）


In [ ]:
df1_local_final.head()

## C=1

In [ ]:
# ===== 先拿 final features，再做 200 萬格點的局部回歸 =====

# 1) 用原本流程先得到 df_all_runs, df_summary（你已經有了）
# df_all_runs, df_summary = run_repeat_screening_stats(...)

# 2) 取 cluster=1 的 final features（用 select_rate=1）
final1_cols_c1 = df1_summary.query("cluster==1 and select_rate==1.0")["feature_idx"].to_numpy(dtype=int)

# 3) 取 cluster=1 的中心點 x_center（沿用你原本挑 x0 的方式，最小改：直接重算一次 x_center）
#    注意：這裡用同一套 pick_x0_indices + x_center 定義，與前面一致
c = 1
idx_c = np.where(c_all == c)[0]
p_hat_c = np.asarray(p_hat_oof)[idx_c]
loc = pick_x0_indices(p_hat_c, target=1, topk=2)
global_ids = [int(idx_c[pos]) for pos in loc]
x_center_c0 = X_all_df.iloc[global_ids].to_numpy().mean(axis=0)

# 4) 用 cluster=0 的最終黑箱模型
m0 = blackbox_by_c[0]

# 5) 抽 200 萬格點做一次 OLS（分批累積）
info_big = local_surrogate_fixed_features_bigN(
    x0=x_center_c0,
    blackbox_model=m0,
    block_cols=final1_cols_c1,
    delta=0.1,
    k=2,
    scaler=None,           # 你現在黑箱吃的是標準化空間
    n_total=2_000_000,
    rng_seed=42,          # 固定即可重現
    center_first=True,
)

# 6) 整理成一張表（係數、p-value、含 intercept）
feature_names = list(X_all_df.columns)
rows = [{
    "feature": "(intercept)",
    "feature_idx": -1,
    "coef": float(info_big["intercept"]),
    "se": np.nan,
    "t": np.nan,
    "p": float(info_big["intercept_p"]),
    "r2": float(info_big["r2"])
}]
for t, col in enumerate(final1_cols_c1):
    rows.append({
        "feature": feature_names[int(col)],
        "feature_idx": int(col),
        "coef": float(info_big["coef_block"][t]),
        "se": float(info_big["se_block"][t]),
        "t": float(info_big["t_block"][t]),
        "p": float(info_big["p_block"][t]),
        "r2": float(info_big["r2"]),
    })

df1_local_final1 = pd.DataFrame(rows).sort_values(["p"], ascending=True).reset_index(drop=True)

# df_local_final 就是：在 x_center_c0 周圍、只用 final features、抽 200 萬格點擬合出的局部線性模型
# y_hat ≈ intercept + Σ coef_j * (x_j - x0_j)   （此處 x_j 指標準化座標；(x_j-x0_j) 就是 X_block）

In [ ]:
df1_local_final1.head()

---
---